# TenderMind على Colab
Runtime > Change runtime type > **T4 GPU**، وبعدين شغّل الخلايا بالترتيب.

In [ ]:
%cd /content
!rm -rf TenderMind && git clone -b stage-5g-production-hardening https://github.com/Younes695/TenderMind.git
%cd /content/TenderMind
!apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y tesseract-ocr tesseract-ocr-ara poppler-utils zstd > /dev/null 2>&1
# install each package; if the pinned version has no wheel for Colab's Python, take the newest one
import subprocess
failed = []
for line in open('requirements.txt'):
    pkg = line.split('#')[0].strip()
    if not pkg:
        continue
    if subprocess.run(['pip', 'install', '-q', '--only-binary=:all:', pkg]).returncode != 0:
        name = pkg.split('==')[0]
        print('pinned failed, installing latest:', name)
        if subprocess.run(['pip', 'install', '-q', name]).returncode != 0:
            failed.append(name)
print('FAILED:', failed if failed else 'none')

In [ ]:
# build the website (React)
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1; apt-get -qq install -y nodejs > /dev/null 2>&1
%cd /content/TenderMind/frontend
!npm ci --silent && npm run build --silent
%cd /content/TenderMind

In [ ]:
# local AI model on the Colab GPU
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
import subprocess, time
subprocess.Popen('nohup ollama serve > /content/ollama.log 2>&1 &', shell=True)
time.sleep(5)
!ollama pull qwen2.5:3b

In [ ]:
# start TenderMind (processing ON - the GPU is Colab's, not your laptop's)
import os, subprocess, time, requests
env = dict(os.environ, TENDERMIND_TWO_STAGE_LLM='1', TENDERMIND_WORKERS_ENABLED='1',
           TENDERMIND_MAX_AI_WORKERS='2', TENDERMIND_ESCALATION_MODEL='off')
subprocess.Popen('cd /content/TenderMind && nohup python -m uvicorn app.main:app --host 0.0.0.0 --port 8001 > /content/app.log 2>&1 &',
                 shell=True, env=env)
time.sleep(10)
try:
    print('server status:', requests.get('http://localhost:8001/', timeout=10).status_code)
except Exception as e:
    print('server not up yet - run the last cell to see the log:', e)

In [ ]:
# public link (temporary - works while this notebook is running)
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
import subprocess, time, re
subprocess.Popen('nohup cloudflared tunnel --url http://localhost:8001 > /content/tunnel.log 2>&1 &', shell=True)
time.sleep(12)
links = re.findall('https://[a-z0-9-]+[.]trycloudflare[.]com', open('/content/tunnel.log').read())
print('OPEN THIS LINK:', links[0] if links else 'not ready - run this cell again')

In [ ]:
# logs if something fails
!tail -n 30 /content/app.log